In [ ]:
import os
import time
import json
import base64
import statistics
from typing import List, Optional

import httpx
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, field_validator
from dotenv import load_dotenv

try:
    import pymupdf as fitz
    PYMUPDF_AVAILABLE = True
except ImportError:
    PYMUPDF_AVAILABLE = False


# 1. ENVIRONMENT & CLIENT SETUP

load_dotenv()  

API_KEY = os.environ.get("Gemini_API_Key")
if not API_KEY:
   
    raise ValueError("CRITICAL ERROR: 'Gemini_API_Key' not found. Set it in your environment or .env file.")

MODEL_FRONTIER = os.environ.get("GEMINI_MODEL_FRONTIER", "gemini-3.8-flash").strip()
MODEL_REASONING = os.environ.get("GEMINI_MODEL_REASONING", "gemini-3.6-flash").strip()
MODEL_OLLAMA = os.environ.get("MODEL_OLLAMA", "my-local-model:latest").strip()
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434").strip()

# gemini exposes an openai-compatible endpoint, so we can reuse the normal openai
# python client instead of installing google's own sdk
gemini_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=API_KEY,
)

# rough per-model pricing (dollars per 1 million tokens) as (input_rate, output_rate)

PRICING = {
    "gemini-3.8-flash": (0.75, 3.75),
    "gemini-3.6-flash": (0.75, 3.75),
    "gemini-3.1-pro-preview": (3.50, 10.50),
    "default": (0.75, 3.75),
}


def calculate_cost(model_id: str, input_tokens: int, output_tokens: int) -> float:
    # falls back to "default" pricing if the model isn't in the dict above
    rates = PRICING.get(model_id, PRICING["default"])
    return (input_tokens / 1_000_000) * rates[0] + (output_tokens / 1_000_000) * rates[1]

# 2. RETRY HELPER (fixes Leg 1's 503)

def call_with_retry(fn, *args, retries: int = 3, base_delay: float = 8.0, **kwargs):
    """Retries on transient 5xx errors with exponential backoff. Does NOT
    retry on a daily quota 429 -- that's a hard cap, not a transient blip,
    so retrying just burns remaining quota."""
    last_err = None
    for attempt in range(retries):
        try:
            # just call whatever function was passed in (e.g. the gemini api call)
            # with whatever arguments were passed in
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            is_daily_quota = "PerDay" in msg
            is_retryable = ("503" in msg or "UNAVAILABLE" in msg) and not is_daily_quota
            if not is_retryable or attempt == retries - 1:
                if is_daily_quota:
                    print("  Hit today's free-tier daily quota (20 requests/day for this model). "
                          "Not retrying -- will reset at midnight Pacific, or enable billing to lift it.")
                raise  
            wait = base_delay * (2 ** attempt)
            print(f"  Transient error ({msg[:80]}...), retrying in {wait:.0f}s "
                  f"(attempt {attempt + 1}/{retries})")
            time.sleep(wait)
            last_err = e
    raise last_err



# 3. PYDANTIC SCHEMAS

# these classes define the exact "shape" every model's response has to match

class ExplanatoryOutput(BaseModel):
    summary: str = Field(description="Core summary of the concept")
    key_points: List[str] = Field(description="Extracted key bullet points")
    confidence_score: float = Field(ge=0.0, le=1.0)  # ge/le = must be between 0 and 1

    @field_validator("key_points", mode="before")
    @classmethod
    def normalize_key_points(cls, v):
        if not isinstance(v, list):
            return v  # not even a list, let pydantic raise its normal error
        normalized = []
        for item in v:
            if isinstance(item, dict):
                name = item.get("name", "")
                desc = item.get("description", "")
                normalized.append(f"{name}: {desc}".strip(": "))
            else:
                normalized.append(str(item))
        return normalized


class DocumentExtractionOutput(BaseModel):
    document_summary: str = Field(description="Overview of document or image content")
    detected_entities: List[str] = Field(description="Names, dates, organizations, identifiers")
    visual_features: List[str] = Field(description="Visual layout or graphical elements")


class ToolFinalOutput(BaseModel):
    # this is the shape we want AFTER the model has used the weather tool
    # and given its final answer back to the user
    answer: str
    tool_used: str
    location: str
    confidence_score: float = Field(ge=0.0, le=1.0)


# 4. BENCHMARK RECORDING

benchmark_results = []  


def record_benchmark(name, provider, model_id, quality, latency, cost, status, notes):
    # just packages everything into one dict so the final table loop is simple
    benchmark_results.append({
        "Category": name, "Provider": provider, "Model": model_id,
        "Quality (Subjective)": quality,
        "Latency": f"{latency:.2f}s" if latency > 0 else "N/A",
        "Cost": f"${cost:.5f}", "Status": status, "Best Use": notes,
    })


# same prompt reused across legs 1-3 
test_prompt = "Explain the difference between a Transformer's encoder and decoder in exactly three sentences."
# tells the model to only output json
sys_json_instr = "You are a rigid data extractor. Reply ONLY with a raw JSON object matching the requested schema. Do not wrap in markdown blocks."

print("NOTE: 15-second delays are enabled to avoid the free tier's per-minute rate limit.\n")


# LEG 1 — FRONTIER

print(f"--- Leg 1: Primary Model ({MODEL_FRONTIER}) ---")
try:
    start = time.perf_counter()  
    # .parse() (instead of .create()) tells the openai sdk to automatically
    # validate the response against our pydantic schema for us
    res = call_with_retry(
        gemini_client.beta.chat.completions.parse,
        model=MODEL_FRONTIER,
        messages=[
            {"role": "system", "content": sys_json_instr},
            {"role": "user", "content": test_prompt},
        ],
        response_format=ExplanatoryOutput,
    )
    latency = time.perf_counter() - start
   
    in_tok = res.usage.prompt_tokens if res.usage else 0
    out_tok = res.usage.completion_tokens if res.usage else 0
    cost = calculate_cost(MODEL_FRONTIER, in_tok, out_tok)
    print(f"Validated Successfully | Latency: {latency:.2f}s | Cost: ${cost:.5f}")
  
    print(res.choices[0].message.parsed.model_dump_json(indent=2))
    record_benchmark("Frontier", "Google", MODEL_FRONTIER, "High", latency, cost, "Validated", "High accuracy structured parsing.")
except Exception as e:
    print(f"API Error: {e}")
    record_benchmark("Frontier", "Google", MODEL_FRONTIER, "N/A", 0, 0, "Error", "Gemini 503/overload — retried and still failed")

time.sleep(15)  # avoid tripping the per-minute rate limit before the next call


# LEG 2 — REASONING

print(f"\n--- Leg 2: Reasoning Model ({MODEL_REASONING}) ---")
try:
    start = time.perf_counter()
    res = call_with_retry(
        gemini_client.beta.chat.completions.parse,
        model=MODEL_REASONING,
        messages=[
            {"role": "system", "content": sys_json_instr},
            {"role": "user", "content": test_prompt},
        ],
        response_format=ExplanatoryOutput,
    )
    latency = time.perf_counter() - start
    in_tok = res.usage.prompt_tokens if res.usage else 0
    out_tok = res.usage.completion_tokens if res.usage else 0
    cost = calculate_cost(MODEL_REASONING, in_tok, out_tok)
    print(f"Validated Successfully | Latency: {latency:.2f}s | Cost: ${cost:.5f}")
    print(res.choices[0].message.parsed.model_dump_json(indent=2))
    record_benchmark("Reasoning", "Google", MODEL_REASONING, "Very High", latency, cost, "Validated", "Complex logical constraints.")
except Exception as e:
    print(f"API Error: {e}")
    record_benchmark("Reasoning", "Google", MODEL_REASONING, "N/A", 0, 0, "Error", str(e)[:60])

time.sleep(15)

# LEG 3 — OLLAMA (pre-flight check + stricter prompt + schema normalization)

print(f"\n--- Leg 3: Local Model via Ollama REST ({MODEL_OLLAMA}) ---")
try:
    # check what models are actually pulled locally BEFORE trying to use one --
    # this turns a confusing 404 into a clear "you haven't pulled this yet" message
    tags_resp = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=10.0)
    tags_resp.raise_for_status()
    available_models = [m["name"] for m in tags_resp.json().get("models", [])]

    if MODEL_OLLAMA not in available_models:
        print(f"'{MODEL_OLLAMA}' is not pulled locally. Available models: {available_models or '(none)'}")
        print(f"Run: ollama pull {MODEL_OLLAMA.split(':')[0]}")
        record_benchmark("Local Open-Weights", "Ollama", MODEL_OLLAMA, "N/A", 0, 0,
                          "Not Found", f"Model not pulled. Available: {available_models or 'none'}")
    else:
        start = time.perf_counter()
        ollama_system_prompt = (
            sys_json_instr + " Your JSON must have exactly these keys: "
            "'summary' (a single string), "
            "'key_points' (a list of PLAIN STRINGS, not objects -- e.g. "
            "[\"encoder handles input context\", \"decoder generates output tokens\"]), "
            "'confidence_score' (a float between 0.0 and 1.0)."
        )

        payload = {
            "model": MODEL_OLLAMA,
            "messages": [
                {"role": "system", "content": ollama_system_prompt},
                {"role": "user", "content": test_prompt},
            ],
            "format": "json",  # tells ollama to force valid json output
            "stream": False,   # we want the whole response at once, not streamed chunks
        }
        response = httpx.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=30.0)
        response.raise_for_status()
        raw_content = response.json()["message"]["content"]
        val = ExplanatoryOutput.model_validate_json(raw_content)
        latency = time.perf_counter() - start
        print(f"Validated Successfully | Latency: {latency:.2f}s | Cost: $0.00")  # local = no api cost
        print(val.model_dump_json(indent=2))
        record_benchmark("Local Open-Weights", "Ollama", MODEL_OLLAMA, "Moderate", latency, 0.0, "Validated", "Zero API cost (excludes hardware).")
except httpx.ConnectError:
    # this specific error means ollama's server isn't even running
    print("Ollama unreachable. Run `ollama serve` or open the Ollama desktop app.")
    record_benchmark("Local Open-Weights", "Ollama", MODEL_OLLAMA, "N/A", 0, 0, "Offline", "Service unreachable")
except ValidationError as e:
    print(f"Validation Error: {e}")
    record_benchmark("Local Open-Weights", "Ollama", MODEL_OLLAMA, "Poor", 0, 0, "Failed Validation", "Missing keys")
except Exception as e:
    print(f"Ollama Error: {e}")
    record_benchmark("Local Open-Weights", "Ollama", MODEL_OLLAMA, "N/A", 0, 0, "Error", str(e)[:60])

time.sleep(15)


# LEG 4 — MULTIMODAL

print(f"\n--- Leg 4: Multimodal Extraction ({MODEL_FRONTIER}) ---")
images_payload = []  # will hold one or more base64-encoded images to send to the model

if os.path.exists("Sample.jpg"):
    with open("Sample.jpg", "rb") as img_file:
        b64_img = base64.b64encode(img_file.read()).decode("utf-8")
        images_payload.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}})

# also generate a fake "invoice" pdf page on the fly and turn it into an image,
# so we always have something to test the pdf-extraction leg with even without
# a real uploaded pdf
if PYMUPDF_AVAILABLE:
    doc = fitz.open()               # creates a blank new pdf in memory
    page = doc.new_page()           # adds one blank page
    page.insert_text((50, 100), "INVOICE #9921\nOrganization: OpenSource Lab\nAmount Due: $450.00\nDate: 2026-03-31", fontsize=14)
    pix = page.get_pixmap()         # renders the page as an image (pixmap)
    b64_pdf = base64.b64encode(pix.tobytes("png")).decode("utf-8")  # convert image to base64 png
    images_payload.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_pdf}"}})
    doc.close()

try:
    if images_payload:
        start = time.perf_counter()
        # the text instruction comes first, then however many images we collected
        content = [{"type": "text", "text": "Extract document summary, detected entities, and visual features from all provided images."}]
        content.extend(images_payload)
        vision_res = call_with_retry(
            gemini_client.beta.chat.completions.parse,
            model=MODEL_FRONTIER,
            messages=[{"role": "user", "content": content}],
            response_format=DocumentExtractionOutput,
        )
        latency = time.perf_counter() - start
        in_tok = vision_res.usage.prompt_tokens if vision_res.usage else 0
        out_tok = vision_res.usage.completion_tokens if vision_res.usage else 0
        cost = calculate_cost(MODEL_FRONTIER, in_tok, out_tok)
        print(f"Validated Visual Extraction | Latency: {latency:.2f}s | Cost: ${cost:.5f}")
        print(vision_res.choices[0].message.parsed.model_dump_json(indent=2))
        record_benchmark("Multimodal", "Google", MODEL_FRONTIER, "High", latency, cost, "Validated", "OCR & Layout Parsing.")
    else:
        print("No images found. Skipping vision leg.")
        record_benchmark("Multimodal", "Google", MODEL_FRONTIER, "N/A", 0, 0, "Skipped", "Missing image files.")
except Exception as e:
    print(f"Multimodal Error: {e}")
    record_benchmark("Multimodal", "Google", MODEL_FRONTIER, "N/A", 0, 0, "Error", str(e)[:60])

time.sleep(15)


# LEG 5 — TOOL LOOP

print(f"\n--- Leg 5: Native Tool Loop ({MODEL_FRONTIER}) ---")


def get_weather(location: str) -> str:

    return f"The weather in {location} is 35°C and sunny."


# this is the schema the model uses to know HOW to call our function --
# it tells the model the function's name, what it does, and what arguments it needs
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string"}},
            "required": ["location"],
        },
    },
}]

# this list is the whole back-and-forth conversation -- it grows as we go
messages = [{"role": "user", "content": "What's the weather in Lahore today?"}]
total_in = 0
total_out = 0

try:
    start = time.perf_counter()
    # step 1: ask the model the question, and tell it which tools it's allowed to use.
    # the model won't actually run the tool itself -- it just tells us it WANTS to
    tool_req = call_with_retry(
        gemini_client.chat.completions.create,
        model=MODEL_FRONTIER, messages=messages, tools=tools,
    )
    if tool_req.usage:
        total_in += tool_req.usage.prompt_tokens
        total_out += tool_req.usage.completion_tokens

    assistant_msg = tool_req.choices[0].message
    messages.append(assistant_msg.model_dump(exclude_none=True))

    if assistant_msg.tool_calls:
        for call in assistant_msg.tool_calls:
            if call.function.name == "get_weather":
                args = json.loads(call.function.arguments)  # arguments come back as a json string
                loc = args.get("location", "Unknown")
                tool_output = get_weather(location=loc)
                messages.append({"role": "tool", "tool_call_id": call.id, "content": tool_output})

        final_res = call_with_retry(
            gemini_client.beta.chat.completions.parse,
            model=MODEL_FRONTIER, messages=messages, response_format=ToolFinalOutput,
        )
        if final_res.usage:
            total_in += final_res.usage.prompt_tokens
            total_out += final_res.usage.completion_tokens

        latency = time.perf_counter() - start
        val = final_res.choices[0].message.parsed
        # cost covers BOTH api calls made in this leg (the tool-call request
        # and the final follow-up request), not just one
        cost = calculate_cost(MODEL_FRONTIER, total_in, total_out)
        print(f"Tool Loop Validated | Latency: {latency:.2f}s | Cost: ${cost:.5f}")
        print(val.model_dump_json(indent=2))
        record_benchmark("Tooling Loop", "Google", MODEL_FRONTIER, "High", latency, cost, "Validated", "Multi-step tool integration.")
    else:
        # model answered directly without needing the weather tool at all
        print("No tool call triggered.")
        record_benchmark("Tooling Loop", "Google", MODEL_FRONTIER, "N/A", 0, 0, "Failed", "No tool call generated.")
except ValidationError as e:
    print(f"Tool loop final validation failed: {e}")
    record_benchmark("Tooling Loop", "Google", MODEL_FRONTIER, "Poor", 0, 0, "Failed Validation", "Schema mismatch.")
except Exception as e:
    print(f"Tool calling pipeline failed: {e}")
    record_benchmark("Tooling Loop", "Google", MODEL_FRONTIER, "N/A", 0, 0, "Error", str(e)[:60])

time.sleep(15)

# LEG 6 — LATENCY / CACHING VARIANCE

print(f"\n--- Leg 6: Prompt Caching / API Variance Analysis ({MODEL_FRONTIER}) ---")
print("NOTE: Gemini's OpenAI-compat endpoint doesn't expose explicit cache-control "
      "or cached_tokens telemetry, so this measures repeated-call latency variance instead.")

# repeat the same long system prompt many times to make it big -- the idea was
# to test prompt caching
long_prompt = "Instruction: Analyze text carefully. " * 500
latencies = []
try:
    for i in range(2): 
        t0 = time.perf_counter()
        r = call_with_retry(
            gemini_client.chat.completions.create,
            model=MODEL_FRONTIER,
            messages=[
                {"role": "system", "content": long_prompt},
                {"role": "user", "content": f"Ping test iteration {i+1}."},
            ],
        )
        latency = time.perf_counter() - t0
        latencies.append(latency)
        in_tok = r.usage.prompt_tokens if r.usage else 0
        out_tok = r.usage.completion_tokens if r.usage else 0
        cost = calculate_cost(MODEL_FRONTIER, in_tok, out_tok)
        print(f"Run {i+1} -> Latency: {latency:.2f}s | Input: {in_tok} | Output: {out_tok} | Cost: ${cost:.5f}")

    print(f"Sample Runs: {[round(x, 2) for x in latencies]}")
    if len(latencies) > 1:
        print(f"Mean Latency: {statistics.mean(latencies):.2f}s | StdDev: {statistics.stdev(latencies):.2f}s")
    else:
        print(f"Mean Latency: {latencies[0]:.2f}s | StdDev: N/A (Insufficient samples)")
except Exception as e:
    print(f"Latency test failed: {e}")


# FINAL REPORT

print("\n--- FINAL COMPARISON TABLE ---")
header = f"| {'Category':<15} | {'Provider':<8} | {'Model':<18} | {'Quality':<10} | {'Latency':<8} | {'Cost':<9} | {'Status':<15} | {'Best Use':<28} |"
print("|" + "-" * (len(header) - 2) + "|")
print(header)
print("|" + "-" * (len(header) - 2) + "|")
for b in benchmark_results:
    print(f"| {b['Category']:<15} | {b['Provider']:<8} | {b['Model']:<18} | {b['Quality (Subjective)']:<10} | "
          f"{b['Latency']:<8} | {b['Cost']:<9} | {b['Status']:<15} | {b['Best Use'][:28]:<28} |")
print("|" + "-" * (len(header) - 2) + "|")

NOTE: 15-second delays are enabled to avoid the free tier's per-minute rate limit.

--- Leg 1: Primary Model (gemini-3.8-flash) ---
  Transient error (Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently e...), retrying in 8s (attempt 1/3)
Validated Successfully | Latency: 36.19s | Cost: $0.00057
{
  "summary": "The encoder processes the entire input sequence simultaneously using bidirectional self-attention to generate rich contextual representations. In contrast, the decoder generates output sequences autoregressively, utilizing masked self-attention to prevent attending to future tokens alongside cross-attention to incorporate the encoder representations. Consequently, encoders are tailored for comprehension tasks like classification, while decoders are specialized for sequential generation tasks like language modeling.",
  "key_points": [
    "Encoders employ bidirectional self-attention, whereas decoders use masked self-attention.",
    "Decoders feature 